# Alkaline platform 270

Every cell here moves a real arm. Two things that are true no matter what the code says:

- **The physical e-stop is the only reliable stop.** The voice stop in the RobotStop
  repo is measured at 42% recall; the software envelope below does not constrain
  joint-space moves at all.
- **`home()` is a joint-space move and nothing checks it.** Its angles were last set
  for a robot swap and drove the arm into the table on 2026-09-16. Raise the arm to a
  clear pose before calling it, and see the warning at `ang_dict` in
  `robotic_testing/common/robotic_arms/xarm7/xarm7_config.py`.

The same arm control is available as a command line tool, which needs no notebook
server: `python robotic_testing/vla.py --help`

## Session

Arm-only session via `init_vla_platform` — no EC-Lab, camera, or TeamViewer/ToDesk
bypasses, none of which the immersion-sweep cells below need. If you need the full
platform (EC-Lab, camera, database) to run `rt.run(...)` electrochemistry, swap this
cell back to `init_alkaline_platform_270`, which needs all of it.

In [ ]:
from robotic_testing.platforms import init_vla_platform

exp_name = 'test'
rt = init_vla_platform(exp_name=exp_name, connect_db=False)
arm = rt.arm

In [ ]:
# start in n hours instead of now
import time

n = 0.1
time.sleep(3600 * n)

## Arm basics

In [ ]:
# Uncomment the one you want. Each is a complete move on its own.

# arm.home()
# arm.move_to_mid_station()
# arm.open_gripper()
# arm.close_gripper()

# where is it now, and is that inside the safety envelope?
# print(arm.describe_state())
# print(arm.describe_envelope())

## Samples

In [ ]:
sample = 0

arm.load_sample(sample)
# arm.unload_sample(sample)

# arm.pick_up_sample(sample)
# arm.put_sample_back(sample)

# a rack position that needs the offset correction applied
# arm.pick_up_sample(11, correction=True)

# a whole rack, one at a time
# for i in range(30, 36):
#     arm.pick_up_sample(i)
#     arm.put_sample_back(i)

## Flask and rinsing

In [ ]:
arm.move_to_flask()

# in, then back out
# arm.sink_in_flask()
# arm.sink_in_flask(reverse=True)

# arm.rinsing()

## Safe immersion sweep, with per-sample dip/hold/lift

Same waypoints as `load_sample`/`unload_sample` — pick up, `mid_station`, above the flask, immersed, `mid_station`, put back — nothing skipped. The final descent and lift segments run at a speed you choose, instead of `sink_in_flask`'s fixed `slow_3` (~12.5 mm/s), and each sample in `SAMPLES` can have its own dip speed, hold time, and lift speed via `per_sample_settings`. Sample rack positions and the flask position are untouched — this only changes speed/timing, never where anything moves to.

Why this is safe: last time's self-collision came from *skipping* `mid_station`, which changed the geometric path enough that the controller picked a different, colliding joint solution. Speed and hold time don't change the path or target pose at all, so they don't carry that risk.

`sink_in_flask_controlled_speed` isn't used here — it looks up `megnan_<speed>` entries in `pos_settings_dict` that only exist for xArm6, not xArm7 (what this platform uses). Speed is set directly instead.

**First real run with a new sample list: try just one or two samples first, hand on the e-stop, before running the full set.**

In [ ]:
# Same waypoints as load_sample/unload_sample (including mid_station), minus rinsing --
# only the final descend/lift segment runs at a chosen speed instead of sink_in_flask's
# fixed slow_3. Sample rack positions and the flask position are untouched throughout --
# only speed and hold time vary per sample.
import time

SAMPLES = [0]                   # sample slots to run, in order. e.g. [0, 1, 2]
DEFAULT_DIP_SPEED_MM_S = 20.0   # descent (immersion) speed, mm/s, for samples not in per_sample_settings
DEFAULT_LIFT_SPEED_MM_S = 20.0  # lift (withdrawal) speed, mm/s, for samples not in per_sample_settings
DEFAULT_HOLD_S = 1              # seconds to hold immersed, 0 = no hold, for samples not in per_sample_settings

# Per-sample overrides: {sample index: [dip speed mm/s, hold seconds, lift speed mm/s]}
# Any sample in SAMPLES that isn't listed here uses the DEFAULT_* values above.
per_sample_settings = {
    # 0: [5.0, 10, 2.0],
}


def sample_settings(sample):
    dip, hold, lift = per_sample_settings.get(
        sample, [DEFAULT_DIP_SPEED_MM_S, DEFAULT_HOLD_S, DEFAULT_LIFT_SPEED_MM_S]
    )
    return dip, hold, lift


cfg = arm.config


def at_speed(mm_s):
    """Default motion settings with the speed swapped to mm_s."""
    if not 0 < mm_s <= 100:
        raise ValueError(f'Speed {mm_s} mm/s must be between 0 and 100')
    return {**cfg.pos_settings_dict['default'], 'speed': mm_s}


def load_sample_at_speed(sample, dip_speed):
    # Same steps as arm.load_sample, just with a chosen speed for the descent.
    arm.pick_up_sample(sample)
    arm.move_to_mid_station()
    arm.move_to_flask()
    arm.move_to_pos(cfg.pos_dict['flask_contact_immersed'], **at_speed(dip_speed))
    arm.sample_in_flask = True


def unload_sample_at_speed(sample, lift_speed):
    # Same steps as arm.unload_sample, minus rinsing, with a chosen speed for the lift-out.
    # move_to_hover_pos does the real distance as an untimed relative move and only
    # applies **kwargs to a second, zero-distance move -- so the speed never took
    # effect there. Doing the lift as one real absolute move fixes that.
    immersed = cfg.pos_dict['flask_contact_immersed']
    above_flask = list(immersed[:2]) + [immersed[2] + cfg.hover_offset_dict['flask']] + list(immersed[3:])
    arm.move_to_pos(above_flask, **at_speed(lift_speed))
    arm.sample_in_flask = False
    arm.move_to_mid_station()
    arm.put_sample_back(sample)


# Print the plan, then wait for confirmation before moving anything.
print('Samples to run:', SAMPLES)
for sample in SAMPLES:
    dip, hold, lift = sample_settings(sample)
    at_speed(dip)   # raises if out of range, before anything moves
    at_speed(lift)
    print(f'  Sample {sample}: descend {dip} mm/s, hold {hold} s, lift {lift} mm/s')
if input('Type yes to start: ').strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')

for sample in SAMPLES:
    dip, hold, lift = sample_settings(sample)

    print(f'[Sample {sample}] load (pick up, move to flask, descend at {dip} mm/s)')
    load_sample_at_speed(sample, dip)

    if hold > 0:
        print(f'[Sample {sample}] hold {hold} s')
        time.sleep(hold)

    print(f'[Sample {sample}] unload (lift at {lift} mm/s, put back, no rinsing)')
    unload_sample_at_speed(sample, lift)

print('Done.')

## Free-form motion

Held to the envelope in the arm's config. A move outside it, or bigger than one step, is
refused before anything moves -- `dry_run=True` reports where a move would end up without
going there.

The envelope is a box with a keep-out cylinder around the base. It constrains
`move_relative` and friends; it does **not** constrain `home()` or any other joint-space
move.

In [ ]:
from robotic_testing.common.robotic_arms.xarm_control import SafetyError

# where would this end up?
print('predicted:', arm.move_relative(dx=30, dy=-20, dz=50, dry_run=True))

# refused before it starts
try:
    arm.move_relative(dz=5000)
except SafetyError as error:
    print('refused: ', error)

# actually move
# arm.move_relative(dx=30, dy=-20, dz=50)
# arm.move_up(40)
# arm.move_down(40)
# arm.move_to_pose(z=300)          # absolute; None keeps the current value
# arm.move_relative_path([[0, 0, 80], [60, 0, 0], [0, 0, -80]], dry_run=True)

## VLA section

This was a hand-written copy of the planner, the pydantic action vocabulary and the
executor. All three now live in `robotic_testing/vla.py` and `common/vla_control.py`,
which is what the command line tool runs and where the envelope checks, the model
fallback chain and the tests are. Imported here rather than maintained twice.

In [ ]:
from robotic_testing.common.vla_control import VLARobot
from robotic_testing.vla import make_planner, handle_instruction, run_plan

vla = VLARobot(arm, logger=getattr(rt, 'logger', None))
planner = make_planner(vla)          # default model chain; make_planner(vla, 'gemini-3.8-flash') to pick one

print(vla.system_prompt()[:400], '...')

In [ ]:
# dry_run=True plans and validates without moving
handle_instruction(vla, planner, 'wave at me', dry_run=True, assume_yes=False)

# handle_instruction(vla, planner, 'lift straight up by 4 cm, then go home', dry_run=True, assume_yes=False)
# handle_instruction(vla, planner, 'grab the beaker on the far side of the bench', dry_run=True, assume_yes=False)

# for real, with a confirmation prompt before anything moves
# handle_instruction(vla, planner, 'wave at me three times', dry_run=False, assume_yes=False)

# a plan written by hand, no planner involved
# run_plan(vla, [{'action': 'wave', 'times': 2}], dry_run=True, assume_yes=False)

## Stopping, and recovering

`emergency_stop()` puts the controller into the stop state, abandoning the current move.
`reset_safety_state()` clears the fault and re-enables motion -- only once the cause is
understood.

In [ ]:
# arm.emergency_stop()
# arm.reset_safety_state()